In [ ]:
%load_ext autoreload
%autoreload 2

# 06 · Comprehensive scoring of M5 forecasts

Score every available forecast against the held-out actuals using:

1. **Bottom-level WRMSSE** — official competition metric at item × store (`m5.evaluation`).
2. **Hierarchical WRMSSE** — re-aggregated across the 12 official M5 hierarchy levels.
3. **Standard accuracy metrics** — MAE, RMSE, sMAPE, signed bias.
4. **Per-series diagnostics** — RMSSE distribution, worst offenders, bias by category/state.
5. **Predicted vs actual** — network total per CV window + top-volume series.

**Inputs**
- `data/processed/long.parquet` — produced by `make prep`.
- `artifacts/cv_*.parquet` — rolling-origin CV outputs from `make cv-stats` / `make cv-lgbm`.
  Schema: `unique_id, ds, y, cutoff, <model_cols...>` (Nixtla convention).

Drop additional CV parquets in `artifacts/` matching `cv_*.parquet` and they are picked up automatically.


In [ ]:
from __future__ import annotations
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from m5.config import SETTINGS
from m5.evaluation import compute_components, wrmsse_for_models
from m5.logging import logger

plt.rcParams.update({"figure.figsize": (12, 5), "figure.dpi": 110, "axes.grid": True})
sns.set_palette("colorblind")

ARTIFACTS = SETTINGS.artifacts_dir
LONG_PARQUET = SETTINGS.processed_dir / "long.parquet"
SCORING_OUT = ARTIFACTS / "scoring_summary.parquet"

print(f"artifacts dir: {ARTIFACTS}")
print(f"long parquet:  {LONG_PARQUET}  exists={LONG_PARQUET.exists()}")


## 1 · Load truth and prediction sets

In [ ]:
if not LONG_PARQUET.exists():
    raise FileNotFoundError(
        f"Run `make prep` first — expected {LONG_PARQUET} but it does not exist."
    )

long = pd.read_parquet(LONG_PARQUET)
keep = ["unique_id", "ds", "y", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
if "sell_price" in long.columns:
    keep.append("sell_price")
long = long[keep].sort_values(["unique_id", "ds"]).reset_index(drop=True)
print(
    f"long: {len(long):,d} rows, "
    f"{long['unique_id'].nunique():,d} series, "
    f"{long['ds'].nunique():,d} days "
    f"({long['ds'].min().date()} → {long['ds'].max().date()})"
)
long.head()


In [ ]:
def discover_prediction_sets(directory: Path, pattern: str = "cv_*.parquet") -> dict[str, Path]:
    found = {p.stem.removeprefix("cv_"): p for p in sorted(directory.glob(pattern))}
    if not found:
        logger.warning(
            f"No prediction sets matched {directory}/{pattern}. "
            "Run `make cv-stats` / `make cv-lgbm` first."
        )
    return found


paths = discover_prediction_sets(ARTIFACTS)
for name, p in paths.items():
    print(f"  {name:<10s} → {p.relative_to(SETTINGS.artifacts_dir.parent)}")


In [ ]:
def load_cv_predictions(paths: dict[str, Path]) -> tuple[pd.DataFrame, list[str]]:
    '''Outer-merge every CV parquet on (unique_id, ds, cutoff). Returns (frame, model_cols).'''
    if not paths:
        return pd.DataFrame(), []
    keys = ["unique_id", "ds", "cutoff"]
    base: pd.DataFrame | None = None
    model_cols: list[str] = []
    for i, (_, path) in enumerate(paths.items()):
        df = pd.read_parquet(path)
        new_cols = [c for c in df.columns if c not in (*keys, "y")]
        model_cols.extend(new_cols)
        if base is None:
            base = df[[*keys, "y", *new_cols]].copy()
        else:
            base = base.merge(df[[*keys, *new_cols]], on=keys, how="outer")
    assert base is not None
    return base, sorted(set(model_cols))


cv, model_cols = load_cv_predictions(paths)
if cv.empty:
    raise RuntimeError("No prediction sets found. Run `make cv-stats` / `make cv-lgbm`.")

print(f"merged eval frame: {len(cv):,d} rows × {cv.shape[1]} cols")
print(f"models:    {model_cols}")
print(f"cutoffs:   {sorted(pd.to_datetime(cv['cutoff'].unique()))}")
horizon_days = (cv['ds'] - cv['cutoff']).dt.days
print(f"horizon:   day {horizon_days.min()} → day {horizon_days.max()}")
cv.head()


## 2 · Bottom-level WRMSSE leaderboard

In [ ]:
# Components are computed on training history strictly *before* the first CV window starts.
# Nixtla's `cutoff` is the last training day; predictions begin at cutoff + 1.
train_cutoff = cv["cutoff"].min()
train_for_components = long[long["ds"] <= train_cutoff].copy()
if train_for_components.empty:
    raise ValueError(f"No training data on or before first CV cutoff {train_cutoff}.")

print(
    f"components computed on {len(train_for_components):,d} rows ending "
    f"{train_for_components['ds'].max().date()} (≤ first cutoff {pd.Timestamp(train_cutoff).date()})"
)

components = compute_components(train_for_components)
print(f"weights: {len(components.weights):,d} series, sum={components.weights.sum():.4f}")
print(f"scales:  {len(components.scales):,d} series (zeros and NaNs dropped)")


In [ ]:
truth = cv[["unique_id", "ds", "y"]].dropna(subset=["y"])
bottom_scores = wrmsse_for_models(truth, cv, components, model_cols=model_cols)

leaderboard = (
    bottom_scores.to_frame("WRMSSE_bottom")
    .assign(rank=lambda d: d["WRMSSE_bottom"].rank(method="min").astype(int))
)
leaderboard


In [ ]:
fig, ax = plt.subplots(figsize=(9, max(3, 0.5 * len(bottom_scores))))
sorted_scores = bottom_scores.sort_values()
sorted_scores.plot.barh(ax=ax, color="steelblue")
ax.set_xlabel("WRMSSE (bottom level — item × store)")
ax.set_ylabel("")
ax.set_title("Bottom-level WRMSSE — lower is better")
for i, v in enumerate(sorted_scores.values):
    ax.text(v, i, f"  {v:.3f}", va="center")
ax.invert_yaxis()
plt.tight_layout()


## 3 · Hierarchical WRMSSE across the 12 M5 levels

`m5.evaluation.wrmsse` only scores the bottom (item × store) level. The official M5 metric is the
average WRMSSE across 12 levels. The helper below re-aggregates truth and forecasts to each level,
sums bottom-level dollar-sales weights into groups (so the weighting stays consistent), and computes
RMSSE on the aggregated series scaled by Naive-1 MSE on the aggregated *training* series.


In [ ]:
M5_LEVELS: list[tuple[str, list[str]]] = [
    ("L1_total",        []),
    ("L2_state",        ["state_id"]),
    ("L3_store",        ["store_id"]),
    ("L4_cat",          ["cat_id"]),
    ("L5_dept",         ["dept_id"]),
    ("L6_state_cat",    ["state_id", "cat_id"]),
    ("L7_state_dept",   ["state_id", "dept_id"]),
    ("L8_store_cat",    ["store_id", "cat_id"]),
    ("L9_store_dept",   ["store_id", "dept_id"]),
    ("L10_item",        ["item_id"]),
    ("L11_item_state",  ["item_id", "state_id"]),
    ("L12_item_store",  ["item_id", "store_id"]),
]

STATIC_COLS = ["item_id", "dept_id", "cat_id", "store_id", "state_id"]
statics = long[["unique_id", *STATIC_COLS]].drop_duplicates("unique_id").reset_index(drop=True)


In [ ]:
def _bottom_dollar_weights(train_long: pd.DataFrame, statics: pd.DataFrame) -> pd.Series:
    sorted_train = train_long.sort_values(["unique_id", "ds"])
    if "sell_price" in sorted_train.columns:
        rev = sorted_train["y"] * sorted_train["sell_price"].fillna(0)
    else:
        rev = sorted_train["y"]
    last28 = (
        sorted_train.assign(_rev=rev)
        .groupby("unique_id", observed=True)
        .tail(28)
        .groupby("unique_id", observed=True)["_rev"]
        .sum()
    )
    return last28.reindex(statics["unique_id"]).fillna(0)


def _agg_key(df: pd.DataFrame, group_cols: list[str]) -> pd.Series:
    if not group_cols:
        return pd.Series("ALL", index=df.index)
    return df[group_cols].astype(str).agg("|".join, axis=1)


def hierarchical_wrmsse(
    train_long: pd.DataFrame,
    cv_long: pd.DataFrame,
    statics: pd.DataFrame,
    levels: list[tuple[str, list[str]]],
    model_cols: list[str],
) -> pd.DataFrame:
    '''WRMSSE at each M5 hierarchy level. Returns DataFrame indexed by level.'''
    bottom_w = _bottom_dollar_weights(train_long, statics)
    static_idx = statics.set_index("unique_id")

    train_with = train_long.merge(statics, on="unique_id", how="left")
    cv_with = cv_long.merge(statics, on="unique_id", how="left")

    rows = []
    for level_name, group_cols in levels:
        agg_label = _agg_key(static_idx, group_cols)
        weights = bottom_w.groupby(agg_label).sum()
        weights = weights / weights.sum()

        tr = train_with.assign(agg_id=_agg_key(train_with, group_cols))
        tr_g = (
            tr.groupby(["agg_id", "ds"], observed=True)["y"].sum()
            .reset_index()
            .sort_values(["agg_id", "ds"])
        )
        tr_g["_diff_sq"] = tr_g.groupby("agg_id", observed=True)["y"].diff().pow(2)
        scales = tr_g.groupby("agg_id", observed=True)["_diff_sq"].mean()
        scales = scales.replace({0.0: np.nan}).dropna()

        cv_part = cv_with.assign(agg_id=_agg_key(cv_with, group_cols))
        cv_truth = cv_part.groupby(["agg_id", "ds"], observed=True)["y"].sum().reset_index()

        scored: dict[str, object] = {"level": level_name, "n_series": int(weights.shape[0])}
        for m in model_cols:
            cv_pred = cv_part.groupby(["agg_id", "ds"], observed=True)[m].sum().reset_index()
            joined = cv_truth.merge(cv_pred, on=["agg_id", "ds"], how="inner")
            err_sq = (joined["y"] - joined[m]).pow(2)
            mse = err_sq.groupby(joined["agg_id"], observed=True).mean()
            common = weights.index.intersection(scales.index).intersection(mse.index)
            rmsse = np.sqrt(mse.loc[common] / scales.loc[common])
            scored[m] = float((weights.loc[common] * rmsse).sum())
        rows.append(scored)

    return pd.DataFrame(rows).set_index("level")


In [ ]:
hier = hierarchical_wrmsse(train_for_components, cv, statics, M5_LEVELS, model_cols)
hier_with_n = hier.copy()
hier_with_n["n_series"] = hier_with_n["n_series"].astype(int)
hier_with_n.style.format({**{m: "{:.4f}" for m in model_cols}, "n_series": "{:d}"}) \
    .background_gradient(cmap="RdYlGn_r", subset=model_cols)


In [ ]:
mean_by_level = hier[model_cols].mean(axis=0).rename("mean_WRMSSE_over_12_levels")
print("Mean WRMSSE across the 12 M5 levels (lower is better):")
print(mean_by_level.sort_values().to_string())

fig, ax = plt.subplots(figsize=(10, max(4, 0.45 * len(M5_LEVELS))))
sns.heatmap(hier[model_cols], annot=True, fmt=".3f", cmap="RdYlGn_r",
            ax=ax, cbar_kws={"label": "WRMSSE"})
ax.set_title("WRMSSE by hierarchy level × model")
ax.set_xlabel("model")
plt.tight_layout()


## 4 · Standard accuracy metrics (MAE, RMSE, sMAPE, signed bias)

In [ ]:
def conventional_metrics(eval_df: pd.DataFrame, model_cols: list[str]) -> pd.DataFrame:
    '''Per-model MAE / RMSE / sMAPE / signed-bias on the bottom level.'''
    rows = []
    e = eval_df.dropna(subset=["y"])
    y_true = e["y"].to_numpy()
    mean_y = float(y_true.mean()) if len(y_true) else np.nan
    for m in model_cols:
        sub = e.dropna(subset=[m])
        yt = sub["y"].to_numpy()
        yh = sub[m].to_numpy()
        err = yh - yt
        denom = np.abs(yt) + np.abs(yh)
        smape = float(np.where(denom > 0, 2 * np.abs(err) / denom, 0.0).mean())
        rows.append({
            "model": m,
            "n_obs": int(len(sub)),
            "MAE": float(np.abs(err).mean()),
            "RMSE": float(np.sqrt((err ** 2).mean())),
            "sMAPE": smape,
            "bias_mean_residual": float(err.mean()),
            "bias_pct_of_mean_y": float(err.mean() / mean_y) if mean_y else np.nan,
        })
    return pd.DataFrame(rows).set_index("model").sort_values("RMSE")


conv = conventional_metrics(cv, model_cols)
conv.style.format({
    "n_obs": "{:,d}",
    "MAE": "{:.3f}",
    "RMSE": "{:.3f}",
    "sMAPE": "{:.2%}",
    "bias_mean_residual": "{:+.3f}",
    "bias_pct_of_mean_y": "{:+.2%}",
}).background_gradient(cmap="RdYlGn_r", subset=["MAE", "RMSE", "sMAPE"])


## 5 · Per-series diagnostics

In [ ]:
def per_series_rmsse(eval_df: pd.DataFrame, components, model_cols: list[str]) -> pd.DataFrame:
    '''RMSSE per (unique_id x model) — feeds the distribution + worst-N analyses.'''
    out = {}
    for m in model_cols:
        merged = eval_df.dropna(subset=["y", m])
        mse = (merged[m] - merged["y"]).pow(2).groupby(merged["unique_id"], observed=True).mean()
        common = mse.index.intersection(components.scales.index)
        out[m] = np.sqrt(mse.loc[common] / components.scales.loc[common])
    return pd.DataFrame(out)


series_rmsse = per_series_rmsse(cv, components, model_cols)
print(f"RMSSE per series — {len(series_rmsse):,d} series × {len(model_cols)} models")
series_rmsse.describe(percentiles=[0.5, 0.75, 0.9, 0.99]).T


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
clip = float(series_rmsse.quantile(0.99).max())
series_rmsse.clip(upper=clip).boxplot(ax=ax, vert=False, showfliers=False)
ax.set_xlabel(f"Per-series RMSSE (clipped at 99th pct = {clip:.2f})")
ax.set_title("Distribution of per-series RMSSE")
plt.tight_layout()


In [ ]:
# Worst 10 series for each model.
top_n = 10
worst = pd.concat(
    {m: series_rmsse[m].sort_values(ascending=False).head(top_n).reset_index(drop=False)
     for m in model_cols},
    axis=1,
)
worst.columns = pd.MultiIndex.from_tuples(worst.columns)
worst


In [ ]:
# Bias by category and state — over- vs under-forecasting per model.
cv_static = cv.merge(statics, on="unique_id", how="left")


def bias_by(group_col: str) -> pd.DataFrame:
    rows = []
    for m in model_cols:
        g = cv_static.dropna(subset=["y", m]).groupby(group_col, observed=True)
        actual = g["y"].mean()
        pred = g[m].mean()
        bias = (pred - actual) / actual.replace(0, np.nan)
        bias.name = m
        rows.append(bias)
    out = pd.concat(rows, axis=1)
    out.index.name = group_col
    return out


bias_cat = bias_by("cat_id")
bias_state = bias_by("state_id")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.heatmap(bias_cat, annot=True, fmt="+.1%", cmap="RdBu_r", center=0, ax=axes[0])
axes[0].set_title("Bias % by category (positive = over-forecast)")
sns.heatmap(bias_state, annot=True, fmt="+.1%", cmap="RdBu_r", center=0, ax=axes[1])
axes[1].set_title("Bias % by state")
plt.tight_layout()


## 6 · Predicted vs actual

In [ ]:
# Network-level (sum across all series) actuals vs forecast for each CV window.
network = cv.groupby(["cutoff", "ds"], observed=True)[["y", *model_cols]].sum().reset_index()

cutoffs = sorted(network["cutoff"].unique())
fig, axes = plt.subplots(len(cutoffs), 1, figsize=(13, 3.4 * len(cutoffs)),
                         sharey=True, squeeze=False)
for ax, cut in zip(axes[:, 0], cutoffs):
    sub = network[network["cutoff"] == cut].sort_values("ds")
    ax.plot(sub["ds"], sub["y"], color="black", lw=2.0, label="actual")
    for m in model_cols:
        ax.plot(sub["ds"], sub[m], lw=1.2, alpha=0.85, label=m)
    ax.set_title(f"Network total — cutoff {pd.Timestamp(cut).date()}")
    ax.legend(loc="upper left", fontsize=8, ncol=2)
plt.tight_layout()


In [ ]:
# Top-K series by trailing dollar-sales weight — predicted vs actual with prior history.
top_k = 5
top_ids = components.weights.sort_values(ascending=False).head(top_k).index.tolist()
print(f"Top {top_k} series by trailing dollar-sales weight: {top_ids}")

history_cutoff = cv["ds"].min() - pd.Timedelta(days=90)
history = long[(long["unique_id"].isin(top_ids)) & (long["ds"] >= history_cutoff)]

fig, axes = plt.subplots(top_k, 1, figsize=(13, 2.6 * top_k), sharex=False)
axes_arr = np.atleast_1d(axes)
for ax, uid in zip(axes_arr, top_ids):
    h = history[history["unique_id"] == uid].sort_values("ds")
    ax.plot(h["ds"], h["y"], color="grey", lw=1.0, alpha=0.5, label="history")
    sub = cv[cv["unique_id"] == uid].sort_values("ds")
    ax.plot(sub["ds"], sub["y"], color="black", lw=1.4, label="actual")
    for m in model_cols:
        ax.plot(sub["ds"], sub[m], lw=1.0, alpha=0.85, label=m)
    ax.set_title(uid)
    ax.legend(loc="upper left", fontsize=8, ncol=2)
plt.tight_layout()


## 7 · Persist scoring summary

In [ ]:
summary = pd.concat(
    {
        "WRMSSE_bottom": bottom_scores,
        "WRMSSE_mean_12L": mean_by_level,
        **{f"WRMSSE_{lvl}": hier.loc[lvl, model_cols] for lvl in hier.index},
        "MAE": conv["MAE"],
        "RMSE": conv["RMSE"],
        "sMAPE": conv["sMAPE"],
        "bias_pct_of_mean_y": conv["bias_pct_of_mean_y"],
    },
    axis=1,
).rename_axis("model").sort_values("WRMSSE_bottom")

ARTIFACTS.mkdir(parents=True, exist_ok=True)
summary.to_parquet(SCORING_OUT)
print(f"wrote {SCORING_OUT}  ({summary.shape[0]} models × {summary.shape[1]} metrics)")
summary.style.format("{:.4f}").background_gradient(cmap="RdYlGn_r")
